# VISCERA — FINAL pipeline: DINOv3 + CG-AMIL (attention-MIL) @448 + SEMI (Run All → ship → submit)

**Recipe (from the SOTA architecture research):**
**DINOv3 ViT-B/16 backbone** → fine-tune last-6 + **WiSE-FT 0.7** + **CG-AMIL head** (gated attention-MIL pooling @ **448** = 784 patches, lifts a few-patch lesion vs mean-pool) + **entropy floor** (anti-collapse) + **SEMI** (Mean-Teacher consistency + one-sided-PU over the ~288k VLM pool) → **12 epochs, 3-seed ensemble + 5-view TTA**.

**Why:** (1) DINOv3 dense features > DINOv2 for patch attention; (2) attention-MIL is the operating-point/**tail lever** (AUROC is high, PPV@90R low → the hardest positives get diluted by mean-pool); (3) SEMI + entropy regularize the attention on 288k unlabeled frames so it survives 127 positives (anti-overfit). No concept Stage-1 for dinov3 (LOCO showed concept-init ≈ null).

**Switch backbone** via `BACKBONE` in the data cell: `dinov3` (default, needs `dinov3.pth` + a one-time gated HF download) or `dinov2` (the exp1 0.018 path with concept Stage-1).

**Run:** cells top-to-bottom → `ship_seed{0,1,2}.pt` (cfg stamps backbone/cg_head/img) → copy to `RARE25-Submission/resources/` → build the offline container (container reconstructs the exact graph from cfg) → submit.

**Drive `DRIVE_DIR`:** data zips, `train.zip`/`val.zip`, `dinov3.pth` (or `dinov2.pth`); optional cached `unl_manifest.npz`. Runtime → **GPU (94GB ideal for 448 + SEMI)**. ⚠️ Set `HF_TOKEN` in the token cell (do NOT commit a real token).

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/HuynhDoTanThanh/RARE2026.git'   # <-- your repo
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# ---- HF token for the gated DINOv3 download (needed ONCE to fetch dinov3.pth; then it caches to Drive) ----
# SECURITY: paste your token HERE in Colab. Do NOT commit a real token to git (this notebook is pushed to GitHub).
import os
os.environ["HF_TOKEN"] = ""   # <-- paste "hf_..." here in Colab, then run. Leave empty in the committed file.
assert os.environ["HF_TOKEN"] or os.path.exists('/content/drive/MyDrive/RARE_LG/dinov3.pth') or True, \
    "set HF_TOKEN (first run) or have dinov3.pth cached on Drive"
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN empty (ok if dinov3.pth already on Drive)")

In [ ]:
# ---- config + data (backbone weights + numbered data zips) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
BACKBONE  = 'dinov3'                            # 'dinov3' (ViT-B/16, stronger dense features + CG-AMIL) | 'dinov2' (exp1 path)
import glob, os, zipfile, shutil
# backbone weights: dinov2.pth (SSL teacher) or dinov3.pth (timm state_dict). Cache on Drive; dinov3 downloaded once.
if BACKBONE == 'dinov2':
    assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
    shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
else:
    if os.path.exists(f'{DRIVE_DIR}/dinov3.pth'):
        shutil.copy(f'{DRIVE_DIR}/dinov3.pth', 'dinov3.pth'); print('REUSED dinov3.pth from Drive')
    else:
        # gated download -> set os.environ['HF_TOKEN']='hf_...' in a cell ABOVE (do NOT commit the token to git)
        import timm, torch
        m = timm.create_model('vit_base_patch16_dinov3', pretrained=True, img_size=448, num_classes=0)
        torch.save(m.state_dict(), 'dinov3.pth'); shutil.copy('dinov3.pth', f'{DRIVE_DIR}/dinov3.pth')
        print('downloaded + cached dinov3.pth')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')))

In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

In [ ]:
# ---- 35-concept target matrix — DINOV2 concept-pretraining only (SKIPPED for dinov3). ----
import os, shutil
if BACKBONE == 'dinov2':
    os.makedirs('phase3/cache', exist_ok=True)
    if os.path.exists('phase3/cache/concept_targets.npz'):
        print('concept_targets.npz present.')
    elif os.path.exists(f'{DRIVE_DIR}/concept_targets.npz'):
        shutil.copy(f'{DRIVE_DIR}/concept_targets.npz', 'phase3/cache/concept_targets.npz'); print('REUSED concept_targets.npz')
    else:
        !python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz
        shutil.copy('phase3/cache/concept_targets.npz', f'{DRIVE_DIR}/concept_targets.npz'); print('built concept_targets.npz')
else:
    print('dinov3 -> no concept-pretraining (fine-tune raw SSL); concept_targets skipped.')

## Stage-1 — concept-supervised pretraining (full-35 losses: certain / uncertain / smoothing)

LIGHT + L2-SP anchor → *shape*, don't overwrite SSL. Upgrades vs the old masked-BCE:
- **`--discrim full15`** — the full discriminative core (9 → 15; adds `color_heterogeneity, whitish_focal_area, vascular_irregularity, dilated_vessels, focal_abnormal_vessels, border_sharpness`). This is the real substance of "more labels".
- **certain** = class-balanced soft-BCE (`--pos_weight_cap 10` rebalances rare-positive concepts, e.g. `depression_ulceration` p=.055 → 10× — the term most likely to move PPV@90R).
- **uncertain** (`--unc_w 0.1`) = prior-pull on `not_assessable` cells instead of hard-masking (never invents a label).
- **smoothing** (`--smooth_eps 0.05`) = target shrinks toward the per-concept prior so the head can't get more confident than the noisy VLM warrants (anti center-memorization).

Every knob **defaults to the old masked-BCE** → clean superset, ablatable via the flags. Note: **7 of the 35 concepts are dead constants** (value ≡ 0: modality/distance/view/landmark/interpretable_fraction/dominant_color/lesion_size) and are auto-dropped; context/acquisition concepts go to a **detached AUX head** so "all 35" never re-injects center style into the trunk.

In [ ]:
# Stage-1: concept-supervised pretraining -> concept_encoder.pt. DINOV2 ONLY (dinov3 fine-tunes RAW SSL).
import os, shutil
if BACKBONE == 'dinov2':
    if os.path.exists(f'{DRIVE_DIR}/concept_encoder.pt'):
        shutil.copy(f'{DRIVE_DIR}/concept_encoder.pt', 'concept_encoder.pt'); print('REUSED concept_encoder.pt -> Stage-1 SKIPPED')
    else:
        print('running Stage-1 pretraining (~15 epochs) ...')
        !python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz \
            --unfreeze 3 --epochs 15 --bs 192 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
            --discrim full15 --context_route detach --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
            --out concept_encoder.pt
        shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt'); print('saved concept_encoder.pt')
else:
    print('dinov3 -> Stage-1 concept-pretraining SKIPPED (fine-tune raw dinov3 + CG-AMIL).')

## Stage-2 — FINAL ship: DINOv3 + CG-AMIL head + 448 + SEMI, 3 seeds

The new solution (from the SOTA architecture research), on a stronger backbone:
- **DINOv3 ViT-B/16** (`--backbone dinov3`): stronger dense/patch features than DINOv2 (Gram anchoring) — directly benefits patch-level attention. Same 5 prefix tokens + 768-d → same head. Fine-tunes RAW DINOv3 (no concept Stage-1; the LOCO gate showed concept-init ≈ null).
- **Image 448** (`featurize.IMG`): 28×28 = 784 patch tokens (vs 576@336) → finer lesion localization.
- **CG-AMIL head** (`--cg-head`): gated attention-MIL pooling (Ilse 2018) instead of mean-pool → a subtle few-patch lesion dominates the pooled vector instead of being diluted (the **tail lever** for PPV@90R). `--attn-entropy 0.02` maximizes attention entropy → **anti-collapse** (can't 1-hot memorize a center cue); the SEMI 288k-pool consistency further regularizes the ~0.2M attention params so it survives 127 positives.
- **SEMI + WiSE-FT 0.7 + 12 epochs + 3-seed + 5-view TTA** as before. WiSE-FT anchors to raw DINOv3.

All train-time; the shipped graph = backbone + attention-pool + LayerNorm + Linear (per-image, no batch stats). `viscera_model.py` reconstructs backbone/pooling/img from the ckpt cfg → verified EXACT parity (Δ=0). Same-center val below = mirage.

In [ ]:
# ---- OPTIONAL de-risk gate (OFF by default) — run ONCE to confirm SEMI doesn't regress before the single submission ----
# Trains SHORT (12ep, no semi) vs SEMI (12ep + semi) with each center held out; paired AUROC/AUPRC gate. Ships NOTHING.
# Leave RUN_GATE=False for the straight Run-All -> ship path. Set True to spend ~4 finetunes (~1h) de-risking SEMI.
RUN_GATE = False
if RUN_GATE:
    import os, shutil
    os.makedirs('phase3/cache', exist_ok=True)
    if not os.path.exists('phase3/cache/unl_manifest.npz'):
        if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
            shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz')
        else:
            !python -m phase3.mine_hardneg --manifest-only
            shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz')
    SHORT = '--unfreeze 6 --wise-ft 0.7 --epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2'
    SEMI  = SHORT + (' --semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
                     '--semi-n 300000 --semi-bs 256 --semi-steps 10')     # == the ship recipe
    for hold in ['center_2', 'center_1']:
        for name, flags in [('short', SHORT), ('semi', SEMI)]:
            print(f'--- holdout {hold} : {name} ---')
            !python -m phase3.finetune --train-csv train_colab.csv --init concept_encoder.pt --seed 0 \
                --holdout {hold} {flags} --out loco_{name}_{hold}.pt
    import numpy as np, phase3.evaluate as ev
    def leg(h, n):
        d = np.load(f'loco_{n}_{h}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
    fails = []
    for hold in ['center_2', 'center_1']:
        y, c, ss = leg(hold, 'semi'); _, _, sh = leg(hold, 'short')
        print(f'\nholdout {hold}  (SEMI vs SHORT):')
        for m in ('auroc', 'auprc'):
            g = ev.gate(y, ss, sh, center=c, metric=m, B=2000)
            if g['verdict'] == 'FAIL': fails.append((hold, m))
            print(f"  {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
    print('\nDECISION:', f'SEMI FAILs {fails} -> ship with --semi-weight 0 (drop SEMI, keep 12ep)' if fails
          else 'SEMI does not regress -> keep it (the ship cell is correct)')
else:
    print('De-risk gate SKIPPED. Ship = 12ep + SEMI. Set RUN_GATE=True to confirm SEMI on the new-center proxy first.')

In [ ]:
# ==== FINAL SHIP: {BACKBONE} + CG-AMIL (attention-MIL@448) + SEMI, 3 seeds -> ensemble ====
import os, shutil
# unl_manifest.npz = paths+suspicion for SEMI (built from the VLM pool; cache to Drive)
if not os.path.exists('phase3/cache/unl_manifest.npz'):
    os.makedirs('phase3/cache', exist_ok=True)
    if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
        shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz'); print('REUSED unl_manifest.npz')
    else:
        !python -m phase3.mine_hardneg --manifest-only
        shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz'); print('built unl_manifest.npz')
INIT = '--init concept_encoder.pt' if BACKBONE == 'dinov2' else ''    # dinov3 fine-tunes RAW SSL (no Stage-1)
# --semi-bs 192 (448 is ~1.8x the 336 memory; raise if VRAM allows). --cg-head = attention-MIL; --attn-entropy anti-collapse.
FLAGS = (f'--backbone {BACKBONE} --cg-head --attn-entropy 0.02 --unfreeze 6 --wise-ft 0.7 --epochs 12 '
         '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
         '--semi-n 300000 --semi-bs 192 --semi-steps 10 --ema-decay 0.99')
for s in [0, 1, 2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none {INIT} --seed {s} \
        {FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt
for s in [0, 1, 2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
if BACKBONE == 'dinov2': shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt')
print(f'DONE -> ship_seed0/1/2.pt ({BACKBONE} + CG-AMIL@448 + SEMI) saved to Drive. Next: Package cell.')
# Safe fallbacks: drop --cg-head (mean-pool) ; drop --semi-manifest (no SEMI) ; --backbone dinov2 --init concept_encoder.pt (exp1 0.018)

## Package the submission (offline container)

`ship_seed{0,1,2}.pt` are the final weights. To submit:
1. Copy them into the container: `RARE25-Submission/resources/ship_seed{0,1,2}.pt`.
2. The container (`model/viscera_model.py`) loads them and runs **5-view TTA (orig/hflip/vflip/rot90/rot270) + 3-seed prob-mean** offline (`--network none`, per-image) — **identical** to the val cell below.
3. Build → test → save: run `do_test_run.sh`, then `do_save.sh`. **Validate the SAVED tar** (`gunzip -c <tar> | docker load` and run that image) before uploading — that's the exact artifact the platform runs.

The val cell below is a **same-center smoke test** = a mirage (optimistic; it read 0.65 before the real 0.018). It only confirms inference works; the honest new-center number is the **leaderboard**.

## Val scoring → competition metric (PPV@90R @1% prevalence, bootstrap median + CI)
Scores `val_colab.csv` with the 3-seed ensemble and reports the leaderboard metric the same way the grader does (curve-point PPV@90R, 1% prevalence, bootstrap), plus AUROC/AUPRC. ⚠️ **This is SAME-CENTER (both centers were in training) → optimistic** (it read 0.65 vs the real new-center 0.018). Use it only as a smoke test that inference works; the honest new-center number comes from **RARE25-val / the leaderboard**, not here. Read the **CI**, not the point.

In [ ]:
# ---- score val with the 3-seed ensemble + 5-VIEW TTA (EXACTLY the shipped container), report the 5 metrics ----
import os, csv, shutil, numpy as np
# restore ensemble from Drive if the runtime was reset
for s in [0, 1, 2]:
    if not os.path.exists(f'ship_seed{s}.pt') and os.path.exists(f'{DRIVE_DIR}/ship_seed{s}.pt'):
        shutil.copy(f'{DRIVE_DIR}/ship_seed{s}.pt', f'ship_seed{s}.pt')
assert all(os.path.exists(f'ship_seed{s}.pt') for s in [0, 1, 2]), 'ship_seed*.pt missing — run the ship cell or copy from Drive'
assert os.path.exists('val_colab.csv'), 'val_colab.csv missing — run the CSV-builder cell (needs out/val)'

# --tta 5view = orig/hflip/vflip/rot90/rot270 + prob-mean = IDENTICAL to RARE25-Submission/model/viscera_model.py,
# so this val number is what the offline container actually produces (not the old hflip-only approximation).
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --csv val_colab.csv --tta 5view --out preds.csv

from phase3 import evaluate as ev
# preds.csv = name,score,label ; pull center from val_colab.csv by frame name
center_by = {os.path.splitext(os.path.basename(r['path']))[0]: r.get('center', '')
             for r in csv.DictReader(open('val_colab.csv'))}
P = list(csv.DictReader(open('preds.csv')))
y = np.array([int(r['label']) for r in P]); s = np.array([float(r['score']) for r in P])
cen = np.array([center_by.get(r['name'], '') for r in P])
print(f'val frames={len(y)}  pos={int(y.sum())}  centers={sorted(set(cen))}  (TTA=5view = the container)\n')

hdr = f"{'split':10s} {'n':>4s} {'pos':>3s} | {'PPV@90R':>8s} {'CI_low':>7s} {'CI_high':>7s} | {'AUROC':>6s} {'AUPRC':>6s}"
print(hdr); print('-' * len(hdr))


def line(tag, yv, sv, cv):
    r = ev.report_full(yv, sv, cv if len(set(cv)) > 1 else None, target=0.9, prevalence=0.01, B=2000)
    print(f"{tag:10s} {r['n']:>4d} {r['pos']:>3d} | {r['ppv90']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f} | "
          f"{r['auroc']:>6.3f} {r['auprc']:>6.3f}")


line('POOLED', y, s, cen)
for cc in sorted(set(cen)):
    mk = cen == cc
    if mk.sum() and y[mk].sum() > 0:
        line(cc, y[mk], s[mk], cen[mk])
print("\nTTA=5view here == the offline container, so val predicts the leaderboard preprocessing faithfully.")
print("PPV@90R = curve-point @1% prevalence (leaderboard metric, HIGH variance at few pos — read the CI).")
print("AUROC/AUPRC = threshold-free ranking, STABLE at few pos, but NOT the 1%-operating-point score.")
print("SAME-CENTER val is optimistic (ship saw both centers); the honest new-center number is RARE25-val / the leaderboard.")